In [43]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

In [44]:
df_walmart = pd.read_csv("../data/Walmart_Store_sales.csv")

df_walmart.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [45]:
df_walmart.shape

(150, 8)

In [46]:
df_walmart.isna().sum()

Store            0
Date            18
Weekly_Sales    14
Holiday_Flag    12
Temperature     18
Fuel_Price      14
CPI             12
Unemployment    15
dtype: int64

In [47]:
round(df_walmart.isna().sum() / df_walmart.shape[0], 2)

Store           0.00
Date            0.12
Weekly_Sales    0.09
Holiday_Flag    0.08
Temperature     0.12
Fuel_Price      0.09
CPI             0.08
Unemployment    0.10
dtype: float64

In [48]:
df_walmart.describe()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
mean,9.866667,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000
max,20.000000,2.771397e+06,1.000000,91.650000,4.193000,226.968844,14.313000


In [49]:
df_walmart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB


Remove outliers from dataset

In [50]:
cols_outlier = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]

for col in cols_outlier:
    mean = df_walmart[col].mean()
    std = df_walmart[col].std()

    lower = mean - 3 * std
    upper = mean + 3 * std

    before = df_walmart.shape[0]
    df_walmart = df_walmart[(df_walmart[col] >= lower) & (df_walmart[col] <= upper)]
    after = df_walmart.shape[0]

    print(f"{col} : {before - after} lines removed")

Temperature : 18 lines removed
Fuel_Price : 13 lines removed
CPI : 8 lines removed
Unemployment : 13 lines removed


In [51]:
df_walmart.shape

(98, 8)

In [52]:
df_walmart["Date"] = pd.to_datetime(df_walmart["Date"], format="%d-%m-%Y", errors="coerce")

df_walmart["year"] = df_walmart["Date"].dt.year
df_walmart["month"] = df_walmart["Date"].dt.month
df_walmart["day"] = df_walmart["Date"].dt.day
df_walmart["day_of_week"] = df_walmart["Date"].dt.day_of_week

df_walmart.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,year,month,day,day_of_week
0,6.0,2011-02-18,1572117.54,NaN,59.61,3.045,214.777523,6.858,2011.0,2.0,18.0,4.0
1,13.0,2011-03-25,1807545.43,0.0,42.38,3.435,128.616064,7.470,2011.0,3.0,25.0,4.0
4,6.0,2010-05-28,1644470.66,0.0,78.89,2.759,212.412888,7.092,2010.0,5.0,28.0,4.0
6,15.0,2011-06-03,695396.19,0.0,69.80,4.069,134.855161,7.658,2011.0,6.0,3.0,4.0
7,20.0,2012-02-03,2203523.20,0.0,39.93,3.617,213.023622,6.961,2012.0,2.0,3.0,4.0


In [53]:
df_walmart.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98 entries, 0 to 149
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         98 non-null     float64       
 1   Date          88 non-null     datetime64[ns]
 2   Weekly_Sales  90 non-null     float64       
 3   Holiday_Flag  87 non-null     float64       
 4   Temperature   98 non-null     float64       
 5   Fuel_Price    98 non-null     float64       
 6   CPI           98 non-null     float64       
 7   Unemployment  98 non-null     float64       
 8   year          88 non-null     float64       
 9   month         88 non-null     float64       
 10  day           88 non-null     float64       
 11  day_of_week   88 non-null     float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 10.0 KB


Drop missing values from the target

In [54]:
df_walmart = df_walmart.dropna(subset=["Weekly_Sales"])

print(df_walmart.shape)
df_walmart.head()

(90, 12)


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,year,month,day,day_of_week
0,6.0,2011-02-18,1572117.54,NaN,59.61,3.045,214.777523,6.858,2011.0,2.0,18.0,4.0
1,13.0,2011-03-25,1807545.43,0.0,42.38,3.435,128.616064,7.470,2011.0,3.0,25.0,4.0
4,6.0,2010-05-28,1644470.66,0.0,78.89,2.759,212.412888,7.092,2010.0,5.0,28.0,4.0
6,15.0,2011-06-03,695396.19,0.0,69.80,4.069,134.855161,7.658,2011.0,6.0,3.0,4.0
7,20.0,2012-02-03,2203523.20,0.0,39.93,3.617,213.023622,6.961,2012.0,2.0,3.0,4.0


EDA

In [55]:
px.histogram(df_walmart, 
             x="Weekly_Sales", 
             nbins=6,
             title="Distribution of Weekly Sales")

The distribution of weekly sales is wide and spread over a large range of values, indicating strong variability across stores and weeks.

In [56]:
px.box(df_walmart, 
       x="Weekly_Sales", 
       title="Distribution of Weekly Sales")

The boxplot highlights a wide variability in weekly sales across stores and weeks.  
The median weekly sales is around 1.2M, while 50% of observations lie between roughly 600k and 1.8M.  
The distribution is right-skewed, indicating that a few weeks reach very high sales levels.

In [57]:
df_date_mean = df_walmart.groupby("Date")["Weekly_Sales"].mean().reset_index()
df_date_mean

,Date,Weekly_Sales
0,2010-02-05,461622.220
1,2010-02-12,1318379.420
2,2010-03-26,1427023.450
3,2010-04-02,561145.140
4,2010-04-16,757738.760
...,...,...
57,2012-06-22,419497.950
58,2012-07-06,1805999.790
59,2012-09-07,597876.550
60,2012-10-12,996978.670


In [58]:
px.line(df_date_mean, x="Date", y="Weekly_Sales", title="Weekly Sales Over Time")

The time series of average weekly sales shows strong week-to-week variability, with no clear long-term trend.  
However, several sharp peaks appear at specific dates, particularly towards the end of each year.  
This suggests that sales are influenced by event-driven factors, such as holiday periods, rather than by a smooth seasonal pattern.

In [59]:
df_month_mean = df_walmart.groupby("month")["Weekly_Sales"].mean().reset_index()
df_month_mean

,month,Weekly_Sales
0,1.0,1.758051e+06
1,2.0,1.414980e+06
2,3.0,1.458858e+06
3,4.0,1.018562e+06
4,5.0,8.311095e+05
5,6.0,1.183841e+06
6,7.0,1.312706e+06
7,8.0,1.231981e+06
8,9.0,9.118161e+05
9,10.0,7.651090e+05


In [60]:
px.bar(df_month_mean, 
       x="month", 
       y="Weekly_Sales",
       title="Average Weekly Sales by Month")

A clear seasonal pattern is observed in weekly sales, with significantly higher average sales in December and at the beginning of the year.  
This suggests that temporal features such as the month of the year are relevant predictors.

In [61]:
df_store_mean = df_walmart.groupby("Store")["Weekly_Sales"].mean().reset_index()
df_store_mean

,Store,Weekly_Sales
0,1.0,1.571115e+06
1,2.0,1.833600e+06
2,3.0,4.090753e+05
3,4.0,2.237004e+06
4,5.0,2.920532e+05
5,6.0,1.588507e+06
6,7.0,5.366644e+05
7,8.0,9.038481e+05
8,9.0,5.060954e+05
9,10.0,1.835589e+06


In [62]:
px.bar(df_store_mean, x="Store", y="Weekly_Sales", title="Average Weekly Sales by Store")

The average weekly sales vary significantly across stores, ranging from a few hundred thousand to more than two million dollars.  
This indicates that the store variable has a strong impact on sales and likely captures differences such as store size or geographical area.

In [63]:
df_holiday_mean = df_walmart.groupby("Holiday_Flag")["Weekly_Sales"].mean().reset_index()
df_holiday_mean

,Holiday_Flag,Weekly_Sales
0,0.0,1.219733e+06
1,1.0,1.225846e+06


In [64]:
px.bar(df_holiday_mean, x="Holiday_Flag", y="Weekly_Sales", title="Average Weekly Sales by Holiday Status")

Holiday periods show only a slight increase in average weekly sales compared to non-holiday weeks.  
The effect remains limited and much weaker than the store effect, which largely dominates sales variability.

In [65]:
cols_corr = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "Weekly_Sales"]

df_corr = df_walmart[cols_corr].corr()
df_corr

,Temperature,Fuel_Price,CPI,Unemployment,Weekly_Sales
Temperature,1.000000,-0.035225,0.155212,-0.201256,-0.091299
Fuel_Price,-0.035225,1.000000,-0.206546,-0.026149,-0.018320
CPI,0.155212,-0.206546,1.000000,-0.205179,-0.373895
Unemployment,-0.201256,-0.026149,-0.205179,1.000000,0.094047
Weekly_Sales,-0.091299,-0.018320,-0.373895,0.094047,1.000000


In [66]:
px.imshow(df_corr,
          title="Impact of Economic Variables on Weekly Sales")

Economic indicators have limited individual linear impact on weekly sales compared to store-specific effects.

Overall, weekly sales are mainly driven by store-specific characteristics, while holidays and economic indicators provide additional but limited explanatory power.  
These insights justify the use of a multivariate linear regression model combining categorical and numerical features.

Preprocessing

In [67]:
print(df_walmart.shape)
df_walmart.columns

(90, 12)


Index(['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature',
       'Fuel_Price', 'CPI', 'Unemployment', 'year', 'month', 'day',
       'day_of_week'],
      dtype='object')

In [68]:
df_walmart.drop(columns="Date", inplace=True)
print(df_walmart.shape)
df_walmart.columns

(90, 11)


Index(['Store', 'Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price',
       'CPI', 'Unemployment', 'year', 'month', 'day', 'day_of_week'],
      dtype='object')

In [69]:
df_walmart.isna().sum()

Store            0
Weekly_Sales     0
Holiday_Flag    10
Temperature      0
Fuel_Price       0
CPI              0
Unemployment     0
year            10
month           10
day             10
day_of_week     10
dtype: int64

In [70]:
df_walmart.info()

<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 149
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         90 non-null     float64
 1   Weekly_Sales  90 non-null     float64
 2   Holiday_Flag  80 non-null     float64
 3   Temperature   90 non-null     float64
 4   Fuel_Price    90 non-null     float64
 5   CPI           90 non-null     float64
 6   Unemployment  90 non-null     float64
 7   year          80 non-null     float64
 8   month         80 non-null     float64
 9   day           80 non-null     float64
 10  day_of_week   80 non-null     float64
dtypes: float64(11)
memory usage: 8.4 KB


In [71]:
df_walmart["Store"] = df_walmart["Store"].astype("object")
df_walmart["Holiday_Flag"] = df_walmart["Holiday_Flag"].astype("object")

df_walmart.info()


<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 149
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         90 non-null     object 
 1   Weekly_Sales  90 non-null     float64
 2   Holiday_Flag  80 non-null     object 
 3   Temperature   90 non-null     float64
 4   Fuel_Price    90 non-null     float64
 5   CPI           90 non-null     float64
 6   Unemployment  90 non-null     float64
 7   year          80 non-null     float64
 8   month         80 non-null     float64
 9   day           80 non-null     float64
 10  day_of_week   80 non-null     float64
dtypes: float64(9), object(2)
memory usage: 8.4+ KB


In [72]:
target = "Weekly_Sales"

x = df_walmart.drop(target, axis=1)
y = df_walmart[target]

print(x)
y

    Store Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment  \
0     6.0          NaN        59.61       3.045  214.777523         6.858   
1    13.0          0.0        42.38       3.435  128.616064         7.470   
4     6.0          0.0        78.89       2.759  212.412888         7.092   
6    15.0          0.0        69.80       4.069  134.855161         7.658   
7    20.0          0.0        39.93       3.617  213.023622         6.961   
..    ...          ...          ...         ...         ...           ...   
139   7.0          0.0        50.60       3.804  197.588605         8.090   
143   3.0          0.0        78.53       2.705  214.495838         7.343   
144   3.0          0.0        73.44       3.594  226.968844         6.034   
145  14.0          0.0        72.62       2.780  182.442420         8.899   
149  19.0          0.0        55.20       4.170  137.923067         8.150   

       year  month   day  day_of_week  
0    2011.0    2.0  18.0          4

0      1572117.54
1      1807545.43
4      1644470.66
6       695396.19
7      2203523.20
          ...    
139     532739.77
143     396968.80
144     424513.08
145    2248645.59
149    1255087.26
Name: Weekly_Sales, Length: 90, dtype: float64

In [73]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

x_train.head()

,Store,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,year,month,day,day_of_week
75,20.0,0.0,75.17,2.808,204.567546,7.856,2010.0,6.0,25.0,4.0
99,13.0,0.0,78.82,2.814,126.139200,7.951,2010.0,7.0,2.0,4.0
120,8.0,0.0,75.32,2.582,214.878556,6.315,2010.0,9.0,17.0,4.0
110,20.0,1.0,28.85,3.179,204.643227,7.484,2010.0,12.0,31.0,4.0
125,3.0,0.0,63.91,3.308,221.643285,7.197,2011.0,11.0,18.0,4.0


In [74]:
numerical_features = x.select_dtypes(exclude="object").columns.tolist()
categorical_features = x.select_dtypes(include="object").columns.tolist()

numerical_imputer = SimpleImputer(strategy="mean")
categorical_imputer = SimpleImputer(strategy="most_frequent")
encoder = OneHotEncoder(drop="first", handle_unknown="ignore")
scaler = StandardScaler()

categorical_pipeline = Pipeline(
    steps=[
        ("cat_imputer", categorical_imputer),
        ("cat_encoder", encoder)
    ]
)

numerical_pipeline = Pipeline(
    steps=[
        ("num_imputer", numerical_imputer),
        ("num_scaler", scaler)
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num_col", numerical_pipeline, numerical_features),
        ("cat_col", categorical_pipeline, categorical_features)
    ]
)

In [75]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

linear_model.fit(x_train, y_train)

y_train_pred = linear_model.predict(x_train)
y_test_pred = linear_model.predict(x_test)

In [76]:
print("R2 score on training set : ", r2_score(y_train, y_train_pred))
print("R2 score on test set : ", r2_score(y_test, y_test_pred))
linear_model

R2 score on training set :  0.9818768409516739
R2 score on test set :  0.9458432628366747


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num_col', ...), ('cat_col', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [77]:
print(y_test[0:5])
y_test_pred[0:5]

59     2020550.99
35     1266564.94
85     1758050.79
114    1757242.51
0      1572117.54
Name: Weekly_Sales, dtype: float64


array([2088557.33969806, 1318216.83777045, 1885056.29356075,
       1391291.20871224, 1658565.09215997])

In [78]:
linear_model.named_steps["regressor"].coef_

array([-3.86622794e+04, -6.56405089e+04,  7.83762929e+05, -1.38158407e+04,
       -2.65924878e+04,  2.83034962e+04, -4.42760132e+04, -2.94530764e-08,
        3.34342099e+05, -1.27885462e+06,  2.42679868e+06, -1.28143087e+06,
        1.09953018e+05, -5.89602501e+05, -7.39623873e+05, -1.19305823e+06,
        2.11168904e+06, -1.75401543e+05,  2.17360707e+06,  1.13311569e+06,
        7.67739291e+05, -5.96860602e+05,  8.69296446e+05,  1.12625854e+06,
        1.48581549e+06,  5.17501355e+05, -8.43349080e+04])

In [79]:
feature_names = linear_model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefs = linear_model.named_steps["regressor"].coef_

df_coef = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
})

df_coef

,feature,coef
0,num_col__Temperature,-3.866228e+04
1,num_col__Fuel_Price,-6.564051e+04
2,num_col__CPI,7.837629e+05
3,num_col__Unemployment,-1.381584e+04
4,num_col__year,-2.659249e+04
5,num_col__month,2.830350e+04
6,num_col__day,-4.427601e+04
7,num_col__day_of_week,-2.945308e-08
8,cat_col__Store_2.0,3.343421e+05
9,cat_col__Store_3.0,-1.278855e+06


In [80]:
df_coef_sorted = df_coef.reindex(
    df_coef["coef"].abs().sort_values(ascending=False).index
)

df_coef_sorted["coef"] = df_coef_sorted["coef"].round(0)
df_coef_sorted

,feature,coef
10,cat_col__Store_4.0,2426799.0
18,cat_col__Store_13.0,2173607.0
16,cat_col__Store_10.0,2111689.0
24,cat_col__Store_19.0,1485815.0
11,cat_col__Store_5.0,-1281431.0
9,cat_col__Store_3.0,-1278855.0
15,cat_col__Store_9.0,-1193058.0
19,cat_col__Store_14.0,1133116.0
23,cat_col__Store_18.0,1126259.0
22,cat_col__Store_17.0,869296.0


In [81]:

px.bar(df_coef_sorted, 
       x="feature", 
       y="coef",
       title="Feature coefficients")

The coefficient analysis shows that store-related features largely dominate the model, reflecting strong differences in baseline sales levels across stores.  
Economic variables such as CPI, fuel price and unemployment have a much smaller impact on weekly sales.  
This confirms that weekly sales are mainly driven by store-specific characteristics rather than economic indicators.

In [82]:
fig = px.scatter(x=y_train, 
                 y=y_train_pred.flatten().tolist(), 
                 title="Actual vs Predicted Weekly Sales (Training Set)")

fig.add_scatter(
    x=y_train,
    y=y_train,
    mode="lines",
    name="True values"
)

fig

In [83]:
fig = px.scatter(x=y_test, 
                 y=y_test_pred.flatten().tolist(), 
                 title="Actual vs Predicted Weekly Sales (Test Set)")

fig.add_scatter(
    x=y_test,
    y=y_test,
    mode="lines",
    name="True values"
)

fig

The baseline linear regression model achieves strong performance on both the training and test sets, with a high R² score and a small gap between them.  
The predicted values closely follow the ideal diagonal line, indicating good predictive accuracy and minimal overfitting.  
However, given the relatively small dataset and the number of features introduced by one-hot encoding, regularized models are explored in the next section to further improve model robustness.

We apply Ridge regression to reduce potential overfitting while preserving all explanatory variables. Ridge limits very large coefficients, encouraging the model to learn smaller coefficients and improving generalization.
We apply Lasso regression to further control overfitting by enforcing a stronger penalty on the coefficients. Lasso tends to shrink less important coefficients to zero, effectively performing feature selection and simplifying the model.
Both Ridge and Lasso regressions are then compared to highlight their differences in terms of coefficient behavior and generalization performance.

In [84]:
ridge_pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge())
    ]
)

params_ridge = {
    "regressor__alpha": np.logspace(-2, 2, 10)
}

ridge_search = GridSearchCV(ridge_pipe, params_ridge, cv=5)
ridge_search.fit(x_train, y_train)


/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown

,estimator,"Pipeline(step...r', Ridge())])"
,param_grid,{'regressor__alpha': array([1.0000...00000000e+02])}
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num_col', ...), ('cat_col', ...)]"


In [85]:
lasso_pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Lasso(max_iter=50000))
    ]
)

params_lasso = {
    "regressor__alpha": np.logspace(0, 2, 10)
}

lasso_search = GridSearchCV(lasso_pipe, params_lasso, cv=5)
lasso_search.fit(x_train, y_train)

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros

/home/olivier/envs/main/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning:

Found unknown

,estimator,Pipeline(step...iter=50000))])
,param_grid,{'regressor__alpha': array([ 1. ...100. ])}
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num_col', ...), ('cat_col', ...)]"


In [86]:
print("Best alpha:", ridge_search.best_params_["regressor__alpha"])

Best alpha: 0.01


In [87]:
best_ridge_model = ridge_search.best_estimator_
best_lasso_model = lasso_search.best_estimator_

print("Ridge R2 score on training set : ", best_ridge_model.score(x_train, y_train))
print("Ridge R2 score on test set : ", best_ridge_model.score(x_test, y_test))

print("Lasso R2 score on training set : ", best_lasso_model.score(x_train, y_train))
print("Lasso R2 score on test set : ", best_lasso_model.score(x_test, y_test))

Ridge R2 score on training set :  0.98116652193537
Ridge R2 score on test set :  0.9539923580396044
Lasso R2 score on training set :  0.9813278963455121
Lasso R2 score on test set :  0.9542984182604412


Both Ridge and Lasso regressions achieve strong and very similar performances, with high R² scores on both training and test sets.  
This confirms that the baseline model already generalizes well. Ridge slightly improves test performance while preserving all features, whereas Lasso does not provide additional benefits in this context.  
Overall, Ridge regression appears more suitable for this dataset, as it effectively reduces potential overfitting while maintaining predictive performance.

In [101]:
ridge_coefs = best_ridge_model.named_steps["regressor"].coef_
lasso_coefs = best_lasso_model.named_steps["regressor"].coef_

df_all_coef = pd.DataFrame({
    "feature": feature_names,
    "ridge_coefs": ridge_coefs.round(0),
    "lasso_coefs": lasso_coefs.round(0),
    "linear_coefs": coefs.round(0)
})

df_all_coef

,feature,ridge_coefs,lasso_coefs,linear_coefs
0,num_col__Temperature,-40756.0,-40677.0,-38662.0
1,num_col__Fuel_Price,-63617.0,-62859.0,-65641.0
2,num_col__CPI,166061.0,235235.0,783763.0
3,num_col__Unemployment,-17343.0,-18770.0,-13816.0
4,num_col__year,21249.0,14172.0,-26592.0
5,num_col__month,36838.0,35624.0,28303.0
6,num_col__day,-45199.0,-44669.0,-44276.0
7,num_col__day_of_week,0.0,0.0,-0.0
8,cat_col__Store_2.0,341187.0,333904.0,334342.0
9,cat_col__Store_3.0,-1196857.0,-1213335.0,-1278855.0


In [104]:
px.bar(
    df_all_coef,
    x="feature",
    y=["ridge_coefs", "lasso_coefs", "linear_coefs"],
    barmode="group",
    title="Model coefficients comparison"
)

As the full coefficient table contains a large number of features, making direct interpretation difficult, the coefficients are sorted by their absolute values in the following table to highlight the most influential features.

In [105]:
top_features = (
    df_all_coef
    .assign(max_abs=df_all_coef[["ridge_coefs", "lasso_coefs", "linear_coefs"]].abs().max(axis=1))
    .sort_values("max_abs", ascending=False)
    .head(10)
)

px.bar(
    top_features,
    x="feature",
    y=["ridge_coefs", "lasso_coefs", "linear_coefs"],
    barmode="group",
    title="Model coefficients comparison (Top 10 features)"
)


Coefficient comparison shows that store-related variables largely dominate the prediction of weekly sales.  
Linear regression produces larger and more extreme coefficients, while Ridge shrinks coefficients smoothly and Lasso removes less informative features by setting some coefficients to zero. Despite these differences, all models capture similar patterns, confirming the robustness of the results.

In [107]:
y_test_pred_ridge = best_ridge_model.predict(x_test)
y_test_pred_lasso = best_lasso_model.predict(x_test)

df_pred = pd.DataFrame({
    "y_true": y_test,
    "Ridge": y_test_pred_ridge,
    "Lasso": y_test_pred_lasso
})

fig = px.scatter(
    df_pred,
    x="y_true",
    y=["Ridge", "Lasso"],
    title="Actual vs Predicted Weekly Sales (Test Set)",
    labels={
        "value": "Predicted Weekly Sales",
        "y_true": "Actual Weekly Sales",
        "variable": "Model"
    }
)

fig.add_scatter(
    x=df_pred["y_true"],
    y=df_pred["y_true"],
    mode="lines",
    name="True values"
)

fig.show()

The comparison between actual and predicted values shows that both Ridge and Lasso models closely follow the ideal diagonal, indicating strong predictive performance.  
The two models produce very similar predictions, which is consistent with their comparable R² scores. This confirms that regularization improves model robustness without degrading performance.

In [108]:
df_pred = pd.DataFrame({
    "y_true": y_test,
    "Linear Regression": y_test_pred,
    "Ridge": y_test_pred_ridge,
    "Lasso": y_test_pred_lasso
})

fig = px.scatter(
    df_pred,
    x="y_true",
    y=["Linear Regression", "Ridge", "Lasso"],
    title="Actual vs Predicted Weekly Sales (Test Set)",
    labels={
        "value": "Predicted Weekly Sales",
        "y_true": "Actual Weekly Sales",
        "variable": "Model"
    }
)

fig.add_scatter(
    x=df_pred["y_true"],
    y=df_pred["y_true"],
    mode="lines",
    name="True values"
)

fig.show()

The comparison between actual and predicted values shows that Linear Regression, Ridge and Lasso models closely follow the reference diagonal.  
All three models produce very similar predictions and closely match the true values, confirming strong predictive performance and good generalization, with Ridge and Lasso providing slightly improved predictions for a few observations.